In [11]:
# pip install pandas numpy matplotlib
# pip install tqdm boto3

import sys
sys.path.append("..")

import pandas as pd
import numpy as np

import io
from tqdm import tqdm
import boto3
from botocore import UNSIGNED
from botocore.config import Config

In [12]:
selected_master_projects = [
    'Business Bay',
    'DAMAC HILLS 2',
    'DAMAC HILLS',
    'TOWN SQUARE',
    'DownTown Dubai',
    'Dubai Creek Harbour',
    'Dubai Marina',
    'Dubai South Residential District',
    'Dubai Sports City',
    'Al Furjan',
    'International City Phase 1',
    'Jumeirah Lakes Towers',
    'Jumeirah Village Circle',
    'Jumeirah Village Triangle',
    'Jumeriah Beach Residence  - JBR',
    'Mudon',
    'Palm Jumeirah',
    'Silicon Oasis',
    'The Greens'
]
print(len(selected_master_projects + ['Global']))

20


### 1 - Get Data

In [13]:

RAW = "s3://ozu-ml-research/realestate_multimodal_ml/transactions/1-transactions-raw-data.csv"
START, END = pd.Timestamp("2014-01-01"), pd.Timestamp("2025-10-01")

bucket = "ozu-ml-research"
key = "realestate_multimodal_ml/transactions/1-transactions-raw-data.csv"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
meta = s3.head_object(Bucket=bucket, Key=key)
total_bytes = meta["ContentLength"]

buffer = io.BytesIO()
with tqdm(total=total_bytes, unit="B", unit_scale=True, desc="Downloading CSV") as pbar:
    s3.download_fileobj(Bucket=bucket, Key=key, Fileobj=buffer, Callback=lambda b: pbar.update(b))

buffer.seek(0)
df = pd.read_csv(buffer)
print(f"Loaded {len(df)} transactions from {RAW}")


Loaded 1577651 transactions from s3://ozu-ml-research/realestate_multimodal_ml/transactions/1-transactions-raw-data.csv


In [14]:
df["date"] = pd.to_datetime(df["instance_date"], dayfirst=True, errors="coerce")

df = df.dropna(subset=["date", "master_project_en"]) # drop either is missing
print(f"Filtered to {len(df)} transactions with valid dates and project names")

df = df[(df["date"] >= START) & (df["date"] <= END)]
print(f"Filtered to {len(df)} transactions between {START} and {END}")

Filtered to 1350591 transactions with valid dates and project names
Filtered to 1032193 transactions between 2014-01-01 00:00:00 and 2025-10-01 00:00:00


In [15]:
# get a fixed full year representation
# raw transactions migh have missing dates (holidays etc.)

full_days = pd.date_range(df["date"].min().normalize(),
                          df["date"].max().normalize(),
                          freq="D")

full_weeks = pd.date_range(df["date"].min().normalize(),
                           df["date"].max().normalize(),
                           freq="W")

print(f"Total days: {len(full_days)}")
print(f"Total weeks: {len(full_weeks)}")

Total days: 4291
Total weeks: 613


### 2 - Prepare Data

In [16]:
print("=== DATASET INSPECTION ===")

print("Total raw rows:", len(df))

mask_year = (df["date"] >= "2014-01-01") & (df["date"] <= "2025-12-31")
df_year = df[mask_year]
print("Rows in 2014–2025:", len(df_year))

df_region = df_year[df_year["master_project_en"].isin(selected_master_projects)]
print("Rows in selected regions:", len(df_region))

print("\nRows per region before cleaning:")
display(df_region.groupby("master_project_en").size().sort_values(ascending=False))

rows_after_trim = {}
for reg in selected_master_projects:
    sub = df_region[df_region["master_project_en"] == reg][["date", "meter_sale_price"]]
    if len(sub) == 0:
        rows_after_trim[reg] = 0
        continue
    q_low, q_high = sub["meter_sale_price"].quantile([0.25, 0.85])
    sub_trim = sub[sub["meter_sale_price"].between(q_low, q_high)]
    rows_after_trim[reg] = len(sub_trim)

print("\nRows per region AFTER quantile trimming:")
display(pd.Series(rows_after_trim).sort_values(ascending=False))

print("\nTotal rows after region + quantile filtering:",
      sum(rows_after_trim.values()))


=== DATASET INSPECTION ===
Total raw rows: 1032193
Rows in 2014–2025: 1032193
Rows in selected regions: 586060

Rows per region before cleaning:


master_project_en
Jumeirah Village Circle             86377
Business Bay                        75990
Dubai Marina                        53629
DownTown Dubai                      46848
Jumeirah Lakes Towers               36653
Al Furjan                           30146
Dubai Sports City                   28459
Palm Jumeirah                       27591
International City Phase 1          27208
Dubai Creek Harbour                 25833
Silicon Oasis                       23825
DAMAC HILLS 2                       23287
TOWN SQUARE                         23164
DAMAC HILLS                         18347
Jumeirah Village Triangle           15060
The Greens                          12490
Dubai South Residential District    11631
Jumeriah Beach Residence  - JBR     11172
Mudon                                8350
dtype: int64


Rows per region AFTER quantile trimming:


Jumeirah Village Circle             51829
Business Bay                        45599
Dubai Marina                        32177
DownTown Dubai                      28109
Jumeirah Lakes Towers               21992
Al Furjan                           18087
Dubai Sports City                   17075
Palm Jumeirah                       16558
International City Phase 1          16324
Dubai Creek Harbour                 15501
Silicon Oasis                       14295
DAMAC HILLS 2                       13972
TOWN SQUARE                         13899
DAMAC HILLS                         11008
Jumeirah Village Triangle            9036
The Greens                           7493
Dubai South Residential District     6978
Jumeriah Beach Residence  - JBR      6703
Mudon                                5015
dtype: int64


Total rows after region + quantile filtering: 351650


#### Prepare Price Indexes

In [17]:
# first, we prepare regional indexes:

train_start = pd.Timestamp("2015-09-01")
train_end = train_start + pd.DateOffset(years=5)

indexed_series = {}
for reg in selected_master_projects:
    # first, we remove the upper and lower quantiles to get a more average representation
    sub = (df[df["master_project_en"] == reg][["date", "meter_sale_price"]].sort_values("date"))

    sub_train = sub[(sub["date"] >= train_start) & (sub["date"] < train_end)]
    q_source = sub_train if len(sub_train) > 0 else sub
    q_low, q_high = q_source["meter_sale_price"].quantile([0.25, 0.85])

    sub = sub[sub["meter_sale_price"].between(q_low, q_high)].reset_index(drop=True)
    # then we calculate the median m2 price on rolling transactions
    # for example median of last 200 transaction
    # we keep the median rolling for each transaction ordered by date
    sub["med200"] = sub["meter_sale_price"].rolling(200).median()
    smooth_w = (sub.dropna(subset=["med200"])
                   .set_index("date")["med200"]
                   .groupby(level=0).last() # here, we take the last value for the day
                   .resample("W").median() # to form a weekly series, resample from daily median
                   .reindex(full_weeks) # and make sure the time-series is calendar complete
                   .ffill() # in case of no transaction in a week, we ffill the rolling price
                   .rolling(4).mean()) # and finally take the moving price of weekly sample
    base_start = pd.Timestamp("2015-09-01")
    first_valid_2015 = smooth_w.loc[base_start:].first_valid_index()
    base_price = smooth_w.loc[first_valid_2015]
    diff_w = smooth_w.diff()
    median52 = diff_w.rolling(52, min_periods=1).median().shift(1)
    mad52 = (diff_w - median52).abs().rolling(52, min_periods=1).median().shift(1)
    k = 3  # tolerance
    capped_diff = diff_w.clip(lower=median52 - k * mad52,
                            upper=median52 + k * mad52).fillna(0)

    capped_diff.loc[:first_valid_2015] = 0
    smooth_w = base_price + capped_diff.cumsum()
    indexed_series[reg] = (smooth_w / base_price * 100).loc[base_start:] # and standardize the price to 100 at 2015.


# secondly, we have a special case for the global index
# using longer sequence for sampling and all regional data together

df_all = (df[df["master_project_en"].isin(selected_master_projects)][["date", "meter_sale_price"]].sort_values("date"))

df_all_train = df_all[(df_all["date"] >= train_start) & (df_all["date"] < train_end)]
q_source_all = df_all_train if len(df_all_train) > 0 else df_all
q_low, q_high = q_source_all["meter_sale_price"].quantile([0.25, 0.85])

df_all = df_all[df_all["meter_sale_price"].between(q_low, q_high)].reset_index(drop=True)
df_all["med600"] = df_all["meter_sale_price"].rolling(600).median()
global_w = (df_all.dropna(subset=["med600"])
               .set_index("date")["med600"]
               .groupby(level=0).last()
               .resample("W").median()
               .reindex(full_weeks)
               .ffill()
               .rolling(4).mean())
base_start = pd.Timestamp("2015-09-01")
first_valid_global_2015 = global_w.loc[base_start:].first_valid_index()
base_price_global = global_w.loc[first_valid_global_2015]
diff_g = global_w.diff()
mean52_g = diff_g.rolling(52, min_periods=1).mean().shift(1)
std52_g  = diff_g.rolling(52, min_periods=1).std(ddof=0).shift(1)
capped_diff_g = diff_g.clip(lower=mean52_g - 2 * std52_g, upper=mean52_g + 2 * std52_g).fillna(0)
capped_diff_g.loc[:first_valid_global_2015] = 0
global_w = base_price_global + capped_diff_g.cumsum()
indexed_series["Global"] = (global_w / base_price_global * 100).loc[base_start:]


#### Prepare Transaction Volume

In [18]:
count_series = {}
for reg in selected_master_projects:
    sub = df[df["master_project_en"] == reg][["date", "meter_sale_price"]]

    weekly_raw = (
        sub.set_index("date")
           .resample("W").size()
           .reindex(full_weeks)
           .fillna(0)
    )

    wk_counts = (
        weekly_raw
        .rolling(8, min_periods=1).mean()
        .loc["2015-09-01":]
    )

    count_series[reg] = wk_counts

df_all = df[df["master_project_en"].isin(selected_master_projects)][["date", "meter_sale_price"]]

global_weekly_raw = (
    df_all.set_index("date")
          .resample("W").size()
          .reindex(full_weeks)
          .fillna(0)
)

global_counts = (
    global_weekly_raw
    .rolling(8, min_periods=1).mean()
    .loc["2015-09-01":]
)
count_series["Global"] = global_counts


#### Save Cleaned Dataset

In [19]:
df_w = pd.DataFrame(indexed_series)                       
counts_df = pd.DataFrame(count_series).reindex(df_w.index).ffill()

print("no null: ", df_w.isna().sum().sum() == 0)

df_w.to_csv("price_series.csv", index_label="date")
counts_df.fillna(0).to_csv("counts_series.csv", index_label="date")

print("Saved.")

no null:  True
Saved.
